# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
# Load data into dbx.default schema
spark.sql("""
    CREATE TABLE IF NOT EXISTS dbx.default.forecast_daily_calendar_imperial AS
    SELECT * FROM samples.accuweather.forecast_daily_calendar_imperial
""")

# Prepare data for regression (example: predicting 'temperature' based on 'day_of_year')
df = spark.sql("SELECT dayofyear(date) AS day_of_year, temperature_avg FROM dbx.default.forecast_daily_calendar_imperial WHERE temperature_avg IS NOT NULL")
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=["day_of_year"], outputCol="features")
df_features = assembler.transform(df)

# Train regression model
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="temperature_avg")
model = lr.fit(df_features)

# Predict future weather (example: for future days)
future_days = spark.createDataFrame([(366,), (367,), (368,)], ["day_of_year"])
future_features = assembler.transform(future_days)
predictions = model.transform(future_features)
display(predictions)

In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Convert Spark DataFrame to pandas and prepare features
pdf = df.toPandas()
X = pdf[["day_of_year"]]
y = pdf["temperature_avg"]

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Custom sklearn pipeline: scaler + Ridge regression (preprocessing encapsulated)
sk_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge",  Ridge(alpha=1.0))
])
sk_pipeline.fit(X_train, y_train)

# Evaluation metrics
y_pred    = sk_pipeline.predict(X_test)
rmse      = np.sqrt(mean_squared_error(y_test, y_pred))
mae       = mean_absolute_error(y_test, y_pred)
r2        = r2_score(y_test, y_pred)
residuals = y_test.values - y_pred

metrics_df = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "R²", "Residual Mean", "Residual Std"],
    "Value":  [
        round(rmse, 4),
        round(mae, 4),
        round(r2, 4),
        round(float(np.mean(residuals)), 4),
        round(float(np.std(residuals)), 4),
    ]
})
display(metrics_df)

In [ ]:
import mlflow
from mlflow.models import infer_signature
import pandas as pd

# ── Custom PyFunc model wrapping the sklearn pipeline ──────────────────────────
class TemperatureForecaster(mlflow.pyfunc.PythonModel):
    """Predicts daily average temperature from day-of-year."""

    def __init__(self, model):
        self._model = model

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        preds = self._model.predict(model_input[["day_of_year"]])
        return pd.DataFrame({"temperature_avg_prediction": preds})

# ── MLflow: log + register to Unity Catalog ───────────────────────────────────
mlflow.set_registry_uri("databricks-uc")
registered_model_name = "dbx.default.temperature_forecaster"

signature = infer_signature(
    X_train,
    pd.DataFrame({"temperature_avg_prediction": sk_pipeline.predict(X_train)})
)

with mlflow.start_run(run_name="temperature_forecast_custom"):
    mlflow.log_params({"model_type": "Ridge", "alpha": 1.0, "test_size": 0.2})
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=TemperatureForecaster(sk_pipeline),
        signature=signature,
        input_example=X_train.head(3),
        registered_model_name=registered_model_name,
    )

print(f"Model registered : {registered_model_name}")
print(f"Model version    : {model_info.registered_model_version}")

In [ ]:
import mlflow

# Register the already-logged model into the workspace (non-UC) model registry
mlflow.set_registry_uri("databricks")

workspace_model_name = "temperature_forecaster"

registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=workspace_model_name,
)

print(f"Workspace registry name : {registered.name}")
print(f"Version                 : {registered.version}")
print(f"Status                  : {registered.status}")